`[1]`

# Baixar mosaicos ASTER + índices espectrais — Colab

Versão Colab de `src/features_process/exportar_mosaicos_indices.py` (e do `.js`
de mesmo nome): exporta para o **Google Drive**, em GeoTIFF, um arquivo por
célula da grid 5×5 e por grupo:

| arquivo | conteúdo | tipo |
|---|---|---|
| `<mosaico>_<LxCy>_bandas` | B01–B14 + bandas de origem do mosaico | Int32 |
| `<mosaico>_<LxCy>_indices` | 21 índices (+ 4 texturas GLCM, opcional) | Float32 |

Mosaicos: `ASTER_mosaico_unico_final` (etapa 3) e `aster-mosaico-final` (v1).
Índices: os de `calculo_spectrals_index.js` (sem repetidos) + `phosphorite_rbd`.
A 30 m, cada célula dá ≈ 0,6–1 GB de bandas e ≈ 1,2 GB de índices.

**Passos:** 2 autenticar → 3 escolher no formulário → 4–5 funções → 6 prévia
(opcional) → 7 plano → 8 iniciar → 9 acompanhar → 10 ver os arquivos no Drive.

In [ ]:
# [2] autenticação — abre um link para autorizar a conta com acesso a mapbiomas-arida
import ee

PROJETO_EE = 'mapbiomas-brazil'  #@param {type:"string"}

ee.Authenticate()
ee.Initialize(project=PROJETO_EE)
print('Earth Engine OK —', PROJETO_EE)

In [ ]:
# [3] escolhas
MOSAICOS_ESCOLHA = 'os dois'  #@param ["os dois", "ASTER_mosaico_unico_final", "aster-mosaico-final"]
GRUPOS_ESCOLHA = 'bandas + índices'  #@param ["bandas + índices", "só bandas", "só índices"]
COM_TEXTURA = False  #@param {type:"boolean"}
CELULAS_TXT = ''  #@param {type:"string"}
ESCALA = 30  #@param [15, 30, 90] {type:"raw"}
PASTA = 'aster_mosaicos_indices'  #@param {type:"string"}
#@markdown `CELULAS_TXT` vazio = as 25 células; ou ex.: `L2C2, L3C1`.

In [ ]:
# [4] parâmetros e índices (iguais aos do .py)
MINE = 'projects/mapbiomas-arida/mine/'
MOSAICOS = {
    'ASTER_mosaico_unico_final': {'asset': MINE + 'ASTER_mosaico_unico_final',
                                  'origem': ['quality', 'data', 'camada', 'swir_estimado', 'normalizacao']},
    'aster-mosaico-final':       {'asset': MINE + 'aster-mosaico-final',
                                  'origem': ['origem', 'swir_estimado']},
}
GRID_ASSET = MINE + 'grid_irece_5x5'      # sem solape: os arquivos não se repetem
BANDAS = ['B01', 'B02', 'B3N', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09',
          'B10', 'B11', 'B12', 'B13', 'B14']
ESCALA_TOA = 1000
ESCALA_PADRAO = 30
PASTA_PADRAO = 'aster_mosaicos_indices'

# TIR: DN → radiância = (DN − 1) × UCC; λ central (µm)
TIR_UCC = {'B10': 6.822e-3, 'B11': 6.780e-3, 'B12': 6.590e-3, 'B13': 5.693e-3, 'B14': 5.225e-3}
TIR_LAMBDA = {'B10': 8.291, 'B11': 8.634, 'B12': 9.075, 'B13': 10.657, 'B14': 11.318}
EPS_MAX = 0.96
C1, C2 = 1.191042e8, 1.4387752e4


def indices_minerais(img):
    b = {n: img.select(n).toFloat() for n in BANDAS}
    return ee.Image.cat([
        b['B02'].divide(b['B01']).rename('ferric_iron_2_1'),
        b['B04'].divide(b['B3N']).rename('ferric_ox_4_3'),
        b['B04'].divide(b['B02']).rename('gossan_4_2'),
        b['B05'].divide(b['B04']).rename('ferrous_sil_5_4'),
        b['B05'].add(b['B07']).divide(b['B06']).rename('sericite_illite_smectite'),
        b['B04'].add(b['B06']).divide(b['B05']).rename('alunite_kaolinite_pyrophyllite'),
        b['B07'].divide(b['B06']).rename('muscovite_7_6'),
        b['B07'].divide(b['B05']).rename('kaolinite_7_5'),
        b['B06'].add(b['B08']).divide(b['B07']).rename('dolomite'),
        b['B13'].divide(b['B14']).rename('carbonate_13_14'),
        b['B14'].divide(b['B12']).rename('quartz_14_12'),
        b['B04'].divide(b['B07']).rename('sultan_r'),
        b['B04'].divide(b['B01']).rename('sultan_g'),
        b['B02'].divide(b['B3N']).multiply(b['B04'].divide(b['B3N'])).rename('sultan_b'),
        b['B04'].divide(b['B3N']).rename('abrams_g'),
        b['B02'].divide(b['B01']).rename('abrams_b'),
    ])


def indices_geo(img):
    n = img.select('B3N').toFloat().divide(ESCALA_TOA)
    r = img.select('B02').toFloat().divide(ESCALA_TOA)
    return ee.Image.cat([
        img.normalizedDifference(['B3N', 'B02']).rename('NDVI'),
        img.expression("(b('B05') + b('B07')) / (2 * b('B06'))").rename('AlOH_Clay'),
        img.expression("b('B08') / b('B06')").rename('Carbonate'),
        n.subtract(r).multiply(1.5).divide(n.add(r).add(0.5)).rename('SAVI'),
    ]).toFloat()


def fosforito_tir(img):
    """(ε11 + ε13) / (2 ε12), emissividade pelo método de normalização."""
    L = {b: img.select(b).toFloat().subtract(1).max(1).multiply(u) for b, u in TIR_UCC.items()}
    T = [L[b].expression('c2 / (lam * log(eps * c1 / (pow(lam, 5) * L) + 1))',
                         {'c1': C1, 'c2': C2, 'lam': TIR_LAMBDA[b], 'eps': EPS_MAX, 'L': L[b]})
         for b in TIR_UCC]
    tmax = ee.Image.cat(T).reduce(ee.Reducer.max())

    def e(b):
        return L[b].divide(tmax.expression('c1 / (pow(lam, 5) * (exp(c2 / (lam * t)) - 1))',
                                           {'c1': C1, 'c2': C2, 'lam': TIR_LAMBDA[b], 't': tmax}))
    return e('B11').add(e('B13')).divide(e('B12').multiply(2)).toFloat().rename('phosphorite_rbd')


def texturas(img):
    b3n = img.select('B3N').unitScale(0, 0.35 * ESCALA_TOA).multiply(255).clamp(0, 255).toByte()
    return (b3n.glcmTexture(size=1)
               .select(['B3N_contrast', 'B3N_var', 'B3N_ent', 'B3N_idm'],
                       ['glcm_contrast', 'glcm_var', 'glcm_ent', 'glcm_hom']).toFloat())


def imagem_indices(img, com_textura=False):
    out = indices_geo(img).addBands(indices_minerais(img)).addBands(fosforito_tir(img))
    return out.addBands(texturas(img)) if com_textura else out

In [ ]:
# [5] export
def tasks_ativas() -> set[str]:
    return {t['description'] for t in ee.data.getTaskList() if t['state'] in ('READY', 'RUNNING')}


def planejar(mosaicos, grupos, celulas, com_textura):
    """Lista de (descrição, imagem, geometria) — nada é iniciado aqui."""
    grid = ee.FeatureCollection(GRID_ASSET)
    plano = []
    for cel in celulas:
        geom = grid.filter(ee.Filter.eq('nome', cel)).geometry()
        for m in mosaicos:
            img = ee.ImageCollection(MOSAICOS[m]['asset']).mosaic().clip(geom)
            base = f'{m}_{cel}'
            if 'bandas' in grupos:
                plano.append((f'{base}_bandas', img.select(BANDAS + MOSAICOS[m]['origem']).toInt32(), geom))
            if 'indices' in grupos:
                plano.append((f'{base}_indices', imagem_indices(img, com_textura), geom))
    return plano


def exportar(plano, escala, pasta):
    ativas = tasks_ativas()
    n = 0
    for desc, img, geom in plano:
        if desc in ativas:
            print(f'  {desc}: já na fila — pulado')
            continue
        task = ee.batch.Export.image.toDrive(
            image=img, description=desc, fileNamePrefix=desc, folder=pasta, region=geom,
            scale=escala, crs='EPSG:4326', maxPixels=1e13, fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True})
        task.start()
        n += 1
        print(f'  {desc}: task {task.id}')
    print(f'{n} tasks iniciadas → Drive/{pasta}')


todas = sorted(ee.FeatureCollection(GRID_ASSET).aggregate_array('nome').getInfo())
mosaicos = list(MOSAICOS) if MOSAICOS_ESCOLHA == 'os dois' else [MOSAICOS_ESCOLHA]
grupos = {'bandas + índices': ['bandas', 'indices'], 'só bandas': ['bandas'],
          'só índices': ['indices']}[GRUPOS_ESCOLHA]
celulas = (todas if not CELULAS_TXT.strip() else
           [c.strip().upper() for c in CELULAS_TXT.replace(';', ',').replace(' ', ',').split(',') if c.strip()])
fora = [c for c in celulas if c not in todas]
assert not fora, f'Células inexistentes: {fora} (válidas: {todas})'
print(f'mosaicos {mosaicos} · grupos {grupos} · {len(celulas)} células · {ESCALA} m · Drive/{PASTA}')

In [ ]:
# [6] prévia no mapa (opcional) — geemap já vem no Colab
import geemap

m = geemap.Map(center=(-11.3, -41.86), zoom=8)
PAL = ['000004', '3b0f70', '8c2981', 'de4968', 'fe9f6d', 'fcfdbf']
for nome in mosaicos:
    img = ee.ImageCollection(MOSAICOS[nome]['asset']).mosaic()
    idx = imagem_indices(img)
    m.addLayer(img, {'bands': ['B04', 'B06', 'B08'], 'min': [167, 75, 45], 'max': [362, 234, 168]},
               f'{nome} — SWIR', False)
    m.addLayer(idx.select(['sultan_r', 'sultan_g', 'sultan_b']),
               {'min': [1.51, 1.51, 0.11], 'max': [2.45, 2.98, 1.20]}, f'{nome} — Sultan RGB', False)
    m.addLayer(idx.select('Carbonate'), {'min': 0.58, 'max': 0.74, 'palette': PAL[::-1]}, f'{nome} — Carbonate', False)
    m.addLayer(idx.select('phosphorite_rbd'), {'min': 0.986, 'max': 1.020, 'palette': PAL},
               f'{nome} — phosphorite_rbd', False)
m.addLayer(ee.Image().paint(ee.FeatureCollection(GRID_ASSET), 0, 1), {'palette': 'ffffff'}, 'Grid 5×5')
m

In [ ]:
# [7] plano — nada é iniciado aqui
plano = planejar(mosaicos, grupos, celulas, COM_TEXTURA)
print(f'{len(plano)} tasks{" (+ texturas)" if COM_TEXTURA else ""}:')
for desc, _, _ in plano:
    print('  ', desc)

In [ ]:
# [8] iniciar as tasks (pula as que já estão na fila)
exportar(plano, ESCALA, PASTA)

In [ ]:
# [9] acompanhar — rode de novo para atualizar
import pandas as pd

descs = {d for d, _, _ in plano}
estado = pd.DataFrame([{'task': t['description'], 'estado': t['state'],
                        'erro': t.get('error_message', '')}
                       for t in ee.data.getTaskList() if t['description'] in descs])
print(estado.estado.value_counts().to_string() if len(estado) else 'nenhuma task encontrada')
estado.drop_duplicates('task').sort_values('task')

In [ ]:
# [10] arquivos no Drive (depois que as tasks terminarem)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
pasta = Path('/content/drive/MyDrive') / PASTA
arqs = sorted(pasta.glob('*.tif')) if pasta.exists() else []
print(f'{len(arqs)} arquivos em {pasta}')
for a in arqs:
    print(f'  {a.name}  {a.stat().st_size / 1e6:,.0f} MB')